# Notebook 03 — Model Training & 4-Baseline Benchmark
**aiPHeed | Week 12 — Member A (W12-3)**

This notebook covers:
1. Walk-forward cross-validation fold structure
2. Four baseline results from `evaluator.py`
3. Baseline comparison + interpretation
4. Targets for LightGBM (W12-2 Member B)

**Training setup:**
- Dataset: `features_fused.parquet` (120 rows: 5 provinces × 24 quarters)
- Labels: `labels.parquet` (label_fies, 50/50 balanced)
- CV: Walk-forward expanding window, min 8 training quarters (2 years)
- Folds: 16 (test quarters: 2022-Q1 → 2025-Q4)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Libraries loaded.')

## 1. Walk-Forward CV Fold Structure

In [ ]:
import pandas as pd
from app.ml.training.cross_validation import WalkForwardSplitter

feat_df = pd.read_parquet('../data/processed/features_fused.parquet')
labels_df = pd.read_parquet('../data/processed/labels.parquet')[['province_code','quarter','label_fies']]
df = feat_df.merge(labels_df, on=['province_code','quarter'])

splitter = WalkForwardSplitter(min_train_quarters=8)
folds = list(splitter.split(df))

print(f'Total folds        : {splitter.n_folds_}')
print(f'Training set sizes : {splitter.fold_metadata_[0]["n_train"]} (fold 1) → '
      f'{splitter.fold_metadata_[-1]["n_train"]} (fold {splitter.n_folds_})')
print(f'Test set size      : {splitter.fold_metadata_[0]["n_test"]} per fold (5 provinces × 1 quarter)')
print(f'First test quarter : {splitter.fold_metadata_[0]["test_quarter"]}')
print(f'Last  test quarter : {splitter.fold_metadata_[-1]["test_quarter"]}')
print(f'Total test rows    : {sum(m["n_test"] for m in splitter.fold_metadata_)}')

In [ ]:
# Visual: fold structure timeline
quarters = sorted(df['quarter'].unique())
q_idx = {q: i for i, q in enumerate(quarters)}

fig, ax = plt.subplots(figsize=(14, 5))

for fold_i, meta in enumerate(splitter.fold_metadata_):
    train_end = q_idx[meta['train_quarters'][-1]]
    test_q = q_idx[meta['test_quarter']]
    # Training bar
    ax.barh(fold_i, train_end + 1, left=0, height=0.7,
            color='#457B9D', alpha=0.7, label='Training' if fold_i == 0 else '')
    # Test bar
    ax.barh(fold_i, 1, left=test_q, height=0.7,
            color='#E63946', alpha=0.9, label='Test' if fold_i == 0 else '')

ax.set_xticks(range(len(quarters)))
ax.set_xticklabels(quarters, rotation=90, fontsize=7)
ax.set_yticks(range(len(splitter.fold_metadata_)))
ax.set_yticklabels([f'Fold {i+1}' for i in range(len(splitter.fold_metadata_))], fontsize=8)
ax.set_xlabel('Quarter')
ax.set_title('Walk-Forward Expanding Window CV\n'
             'Blue = training window (grows each fold)   Red = test quarter (strictly future)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../data/processed/fig_walk_forward_folds.png', bbox_inches='tight')
plt.show()
print('Saved: data/processed/fig_walk_forward_folds.png')

## 2. Load Baseline Results

In [ ]:
with open('../data/processed/eval_results.json', encoding='utf-8') as f:
    results = json.load(f)

BASELINE_DISPLAY = {
    'naive_persistence'    : 'Naive Persistence',
    'psa_only_logistic'    : 'PSA-Only Logistic',
    'nlp_only_logistic'    : 'NLP-Only Logistic',
    'full_feature_logistic': 'Full-Feature Logistic',
}

rows = []
for key, label in BASELINE_DISPLAY.items():
    ov = results['baselines'][key]['overall']
    agg = results['baselines'][key]['aggregate_across_folds']
    rows.append({
        'Baseline'         : label,
        'Weighted F1'      : f"{ov['weighted_f1']:.3f}",
        'ROC-AUC'          : f"{ov['roc_auc']:.3f}",
        'Accuracy'         : f"{ov['accuracy']:.3f}",
        'F1 mean ± std'    : f"{agg['weighted_f1_mean']:.3f} ± {agg['weighted_f1_std']:.3f}",
        'AUC mean ± std'   : f"{agg['roc_auc_mean']:.3f} ± {agg['roc_auc_std']:.3f}",
    })

summary_df = pd.DataFrame(rows).set_index('Baseline')
print('=== 4-Baseline Benchmark Results ===')
print(summary_df.to_string())
print()
print(f"LightGBM targets: Weighted F1 ≥ {results['target_metrics']['lightgbm_weighted_f1_target']} | "
      f"ROC-AUC ≥ {results['target_metrics']['lightgbm_roc_auc_target']}")

## 3. Baseline Comparison Chart

In [ ]:
baseline_keys = list(BASELINE_DISPLAY.keys())
baseline_labels = list(BASELINE_DISPLAY.values())

f1_vals  = [results['baselines'][k]['overall']['weighted_f1'] for k in baseline_keys]
auc_vals = [results['baselines'][k]['overall']['roc_auc']     for k in baseline_keys]
acc_vals = [results['baselines'][k]['overall']['accuracy']    for k in baseline_keys]

f1_std  = [results['baselines'][k]['aggregate_across_folds']['weighted_f1_std'] for k in baseline_keys]
auc_std = [results['baselines'][k]['aggregate_across_folds']['roc_auc_std']     for k in baseline_keys]

x = np.arange(len(baseline_labels))
width = 0.28

fig, ax = plt.subplots(figsize=(11, 5))

bars1 = ax.bar(x - width, f1_vals,  width, label='Weighted F1',  color='#457B9D',
               yerr=f1_std, capsize=4, alpha=0.85)
bars2 = ax.bar(x,         auc_vals, width, label='ROC-AUC',      color='#2A9D8F',
               yerr=auc_std, capsize=4, alpha=0.85)
bars3 = ax.bar(x + width, acc_vals, width, label='Accuracy',     color='#E9C46A', alpha=0.85)

# Target lines
ax.axhline(0.75, color='#E63946', linestyle='--', linewidth=1.2,
           label='LightGBM target F1 ≥ 0.75')
ax.axhline(0.80, color='#F4A261', linestyle='--', linewidth=1.2,
           label='LightGBM target AUC ≥ 0.80')

# Value annotations
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Score')
ax.set_title('4-Baseline Benchmark on Walk-Forward CV Folds\n'
             'Error bars = ±1 std across 16 folds')
ax.set_xticks(x)
ax.set_xticklabels(baseline_labels, rotation=0, fontsize=9)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8, loc='lower right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/fig_baseline_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: data/processed/fig_baseline_comparison.png')

## 4. Per-Fold F1 over Time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

COLORS_B = {'naive_persistence':'#E63946','psa_only_logistic':'#457B9D',
            'nlp_only_logistic':'#2A9D8F','full_feature_logistic':'#E9C46A'}

for key, label in BASELINE_DISPLAY.items():
    fold_f1 = [fm['weighted_f1'] for fm in results['baselines'][key]['fold_metrics']]
    test_qs = [fm['test_quarter'] for fm in results['baselines'][key]['fold_metrics']]
    ax.plot(test_qs, fold_f1, marker='o', markersize=4,
            label=label, color=COLORS_B[key], linewidth=1.5)

ax.axhline(0.75, color='gray', linestyle='--', linewidth=0.8, label='Target F1=0.75')
ax.set_title('Per-Fold Weighted F1 by Test Quarter')
ax.set_xlabel('Test Quarter')
ax.set_ylabel('Weighted F1')
ax.tick_params(axis='x', rotation=45)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/fig_fold_f1_over_time.png', bbox_inches='tight')
plt.show()

## 5. Interpretation

| Baseline | Weighted F1 | ROC-AUC | Interpretation |
|---|---|---|---|
| Naive Persistence | ~0.875 | ~0.867 | Food insecurity state is **persistent** — once a province is at HIGH risk, it tends to stay there. This is the hardest baseline to beat. |
| PSA-Only Logistic | ~0.851 | ~0.833 | Economic/climate features capture the label well. Confirms PSA indicators are informative. |
| NLP-Only Logistic | ~0.482 | ~0.445 | **Low** because FSSI=0.0 (keyword fallback). With transformer FSSI scores, this would improve substantially. |
| Full-Feature Logistic | ~0.851 | ~0.842 | Similar to PSA-only — NLP features diluted by zeros. Fusion will improve once transformer runs. |

### LightGBM Targets (W12-2 — Member B)
- **Weighted F1 ≥ 0.75** — must beat PSA-only logistic
- **ROC-AUC ≥ 0.80** — must beat PSA-only logistic
- **Must beat naive persistence** on F1 to demonstrate added value over a simple carry-forward rule

### Important Note on FSSI
The NLP-only baseline underperforms because all FSSI values are 0.0 (keyword fallback used during Week 11 because XLM-RoBERTa was not cached). Once `--use-transformer` is run, FSSI scores will be populated and the NLP contribution will increase. Member B's LightGBM training should proceed with the current parquets; the transformer run can be done as a follow-up to validate NLP feature importance.

### Leakage Note
`pct_total_hunger` (SWS hunger rate) is **excluded** from all feature sets because the label `label_fies` is defined as `sws_hunger_t > cycle_median`. Including it would be direct label leakage.

## 6. W12-3 Summary

| Output | Status |
|---|---|
| `app/ml/training/cross_validation.py` | ✓ WalkForwardSplitter implemented |
| `app/ml/training/evaluator.py` | ✓ 4 baselines benchmarked |
| `data/processed/eval_results.json` | ✓ Saved |
| 16 walk-forward folds | ✓ min_train=8 quarters |
| NLP features | ⚠ FSSI=0.0 (keyword fallback) — transform run needed |
| `notebooks/03_model_training.ipynb` | ✓ This notebook |

**Ready for W12-2 (Member B): trainer.py + Optuna + lgbm_best.pkl**